# 🐾 MiniProject. Oxford-IIIT Pet — CNN 종합 실습
## CNN 구조와 영상 데이터 처리

---
### 📌 이 노트북의 사용 방법

이 노트북은 **빈칸 채우기가 아닌 직접 코드를 작성**하는 방식으로 구성되어 있습니다.

각 단계의 마크다운 셀에는 **무엇을 구현해야 하는지** + **어느 섹션 코드를 참고하면 되는지** 안내가 있습니다.  
섹션 실습 노트북을 열어두고 함께 참고하며 작성하세요.

| Option | 태스크 | 참고 실습 노트북 |
|--------|--------|----------------|
| **A** | 품종 분류 | `Section05_실습.ipynb`, `Section06_실습.ipynb` |
| **B** | 전경/배경 분할 | `Section07_실습.ipynb`, `Section08_실습.ipynb` |
| **C** | 머리 영역 탐지 | `Section09_실습.ipynb` |

### 📦 데이터셋
| 항목 | 내용 |
|------|------|
| 이름 | Oxford-IIIT Pet |
| 크기 | 약 7,400장 |
| 클래스 | 37종 (개 25 + 고양이 12) |
| 어노테이션 | 분류 레이블 + 분할 마스크 + 머리 BBox |

> 💡 **처음부터 완벽할 필요 없습니다.**  
> 한 번이라도 끝까지 실행해본 경험이 가장 중요합니다.

## ⚙️ 환경 설정

> 아래 세 셀을 순서대로 실행하세요. 수정 불필요합니다.

In [ ]:
# ⚙️ 환경 설정 — 반드시 가장 먼저 실행하세요
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
import torchvision.models as models
from torchvision.datasets import OxfordIIITPet
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
)
from torchvision.ops import box_iou, nms
from torch.utils.data import DataLoader, Dataset, Subset
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import random
from pathlib import Path
from collections import Counter
from sklearn.metrics import confusion_matrix
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42); np.random.seed(42); random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_ROOT = './data'
print(f"✅ 디바이스:    {device}")
print(f"✅ PyTorch:     {torch.__version__}")
print(f"✅ torchvision: {torchvision.__version__}")

In [ ]:
# 1. 폰트 설치
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv

# 2. Matplotlib 캐시 삭제 (중요: 이걸 해야 설치된 폰트를 인식합니다)
import matplotlib as mpl
import matplotlib.font_manager as fm
!rm -rf ~/.cache/matplotlib

# 3. 설치된 폰트 경로 확인 및 등록
for font_info in fm.findSystemFonts(fontpaths=None, fontext='ttf'):
    if 'Nanum' in font_info:
        fm.fontManager.addfont(font_info)

import matplotlib       # 그래프 기반 라이브러리 (설정용)
import matplotlib.pyplot as plt   # 실제 그래프 그리기 모듈
import seaborn as sns   # 통계 특화 시각화

# ── 한글 폰트 설정 ───────────────────────────────────────────────
# matplotlib의 기본 폰트는 한글을 지원하지 않아 '□□□'처럼 깨집니다
# 'NanumGothic'은 Google Colab에 기본으로 설치된 한글 폰트입니다
plt.rc('font', family='NanumBarunGothic')
# axes.unicode_minus=False : 마이너스(-) 기호가 깨지는 문제 방지
plt.rcParams['axes.unicode_minus'] = False
# seaborn 테마 설정: 'whitegrid' = 흰색 배경 + 격자선
sns.set_theme(style='whitegrid')
# Seaborn 사용하는 경우
sns.set(font="NanumBarunGothic",
        rc={"axes.unicode_minus": False},
        style='whitegrid')

In [ ]:
# ✅ 로컬 방식
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

nanum_fonts = [f for f in fm.fontManager.ttflist if 'Nanum' in f.name]

if nanum_fonts:
    font_path = nanum_fonts[0].fname          # 폰트 파일 경로 추출
    font_name = fm.FontProperties(fname=font_path).get_name()  # 실제 등록 이름 확인
    plt.rcParams['font.family'] = font_name
    print(f"✅ 적용 폰트: {font_name}")
else:
    plt.rcParams['font.family'] = 'Malgun Gothic'
    print("⚠️ 나눔 폰트 없음 → 맑은 고딕 적용")

plt.rcParams['axes.unicode_minus'] = False

---
## STEP 1. 데이터 탐색 (EDA)

아래 코드는 모두 제공됩니다. 실행하며 데이터 구조를 파악하세요.

> 💡 **탐색 체크리스트**
> - 37종 품종이 어떻게 생겼는지 확인한다
> - 클래스별 샘플 수 균형을 파악한다
> - 분할 마스크의 레이블(1=전경, 2=배경, 3=경계) 구조를 이해한다
> - BBox가 어떤 영역(머리)을 나타내는지 확인한다

In [ ]:
import os
import torch
import xml.etree.ElementTree as ET
from PIL import Image
from torch.utils.data import Dataset
import torchvision.transforms as transforms

class PetDetectionDataset(Dataset):
    def __init__(self, root, split='trainval', transform=None):
        self.root = root
        self.transform = transform
        # torchvision이 다운로드한 폴더 구조에 맞춤
        self.img_dir = os.path.join(root, 'oxford-iiit-pet', 'images')
        self.anno_dir = os.path.join(root, 'oxford-iiit-pet', 'annotations', 'xmls')

        # trainval.txt 또는 test.txt 파일에서 이미지 이름 목록 읽기
        split_file = os.path.join(root, 'oxford-iiit-pet', 'annotations', f'{split}.txt')
        with open(split_file, 'r') as f:
            self.file_names = [line.strip().split()[0] for line in f.readlines()]

        # ⚠️ Oxford Pet 데이터셋은 일부 이미지에 XML(바운딩 박스) 파일이 누락되어 있습니다.
        # 따라서 XML 파일이 실제로 존재하는 이미지만 필터링해야 합니다.
        self.file_names = [f for f in self.file_names if os.path.exists(os.path.join(self.anno_dir, f'{f}.xml'))]

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, idx):
        file_name = self.file_names[idx]

        # 1. 이미지 로드
        img_path = os.path.join(self.img_dir, f'{file_name}.jpg')
        img = Image.open(img_path).convert("RGB")

        # 2. XML (BBox) 파싱
        xml_path = os.path.join(self.anno_dir, f'{file_name}.xml')
        tree = ET.parse(xml_path)
        root_xml = tree.getroot()

        # BBox 좌표 추출 (PASCAL VOC 형식)
        bndbox = root_xml.find('object').find('bndbox')
        xmin = int(bndbox.find('xmin').text)
        ymin = int(bndbox.find('ymin').text)
        xmax = int(bndbox.find('xmax').text)
        ymax = int(bndbox.find('ymax').text)

        # Faster R-CNN 모델이 요구하는 타겟 형식(Dict)으로 구성
        boxes = torch.tensor([[xmin, ymin, xmax, ymax]], dtype=torch.float32)
        target = {
            "boxes": boxes,
            "labels": torch.ones((1,), dtype=torch.int64), # 단일 클래스 (머리=1)
            "image_id": torch.tensor([idx])
        }

        if self.transform is not None:
            img = self.transform(img)

        return img, target

pet_cls = OxfordIIITPet(root=DATA_ROOT, split='trainval',
                         target_types='category', download=True,
                         transform=transforms.ToTensor())
pet_seg = OxfordIIITPet(root=DATA_ROOT, split='trainval',
                         target_types='segmentation', download=True,
                         transform=None, target_transform=None)
pet_det = PetDetectionDataset(
    root=DATA_ROOT,
    split='trainval',
    transform=transforms.ToTensor()
)

CLASS_NAMES = pet_cls.classes
print(f"클래스 수: {len(CLASS_NAMES)}  |  전체 샘플: {len(pet_cls)}")
print(f"클래스 예시: {CLASS_NAMES[:5]}...")

In [ ]:
# STEP 1-2. 클래스별 샘플 시각화
fig, axes = plt.subplots(4, 9, figsize=(18, 9))
fig.suptitle('Oxford-IIIT Pet — 37종 품종 샘플', fontsize=13, fontweight='bold')

class_samples = {}
for img, label in pet_cls:
    if label not in class_samples: class_samples[label] = img
    if len(class_samples) == 37: break

for idx, (label, img) in enumerate(sorted(class_samples.items())):
    row, col = divmod(idx, 9)
    if row < 4:
        ax = axes[row][col]
        ax.imshow(img.permute(1,2,0).clamp(0,1))
        ax.set_title(CLASS_NAMES[label].replace('_',' '), fontsize=6, pad=2)
        ax.axis('off')
for idx in range(len(class_samples), 36):
    row, col = divmod(idx, 9)
    if row < 4: axes[row][col].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# STEP 1-3. 클래스별 샘플 수 분포
label_counts = Counter(label for _, label in pet_cls)
sorted_items = sorted(label_counts.items())
names  = [CLASS_NAMES[l].replace('_',' ') for l,_ in sorted_items]
counts = [c for _,c in sorted_items]

fig, ax = plt.subplots(figsize=(16, 4))
colors = ['#4fc3f7' if i < 25 else '#ff8a65' for i in range(37)]
ax.bar(range(37), counts, color=colors, alpha=0.8, edgecolor='none')
ax.set_xticks(range(37)); ax.set_xticklabels(names, rotation=45, ha='right', fontsize=7)
ax.axhline(np.mean(counts), color='red', lw=1.5, ls='--', label=f'평균 {np.mean(counts):.0f}장')
ax.set_title('클래스별 샘플 수 (파란색=개 / 주황색=고양이)', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()
print(f"최소: {min(counts)}장 / 최대: {max(counts)}장 / 평균: {np.mean(counts):.0f}장")

---
## STEP 2. 태스크 선택 및 근거 작성 ✏️ 직접 작성

아래 마크다운 셀에 선택한 태스크와 근거를 작성하세요.

| 옵션 | 태스크 | 참고 섹션 |
|------|--------|-----------|
| **A** | 품종 분류 | Section 05, 06 |
| **B** | 전경/배경 분할 | Section 07, 08 |
| **C** | 머리 영역 탐지 | Section 09 |

### 📝 나의 선택

**선택한 태스크**: (A / B / C 중 하나를 적으세요)

**선택 근거**:
- (왜 이 태스크를 골랐는지 2~3문장으로 작성하세요)

**목표 성능**:
- (도달하고 싶은 정량 목표: 예) Accuracy > 70%, mIoU > 0.5)

---
## STEP 3. 전처리 파이프라인 구성 ✏️ 직접 작성

### 📖 참고: `Section04_실습.ipynb` — 실습 3. 훈련/검증용 Transform 파이프라인

선택한 태스크에 맞는 전처리 파이프라인을 작성하세요.

---
**Option A (분류)** 구성 힌트:
```
훈련용: Resize(224,224) → RandomHorizontalFlip → ColorJitter → ToTensor → Normalize
검증용: Resize(224,224) → ToTensor → Normalize
ImageNet 정규화값: mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]
```
DataLoader 분할: train 70% / val 15% / test 15%

---
**Option B (분할)** 구성 힌트:
```
# 이미지와 마스크에 반드시 동일한 변환을 적용해야 합니다 (Section07 실습 참고)
# torchvision.transforms.functional (TF) 을 사용하세요
# 마스크 레이블 재매핑: 1→0(전경), 2→1(배경), 3→2(경계)
# IMG_SIZE = 128
```

---
**Option C (탐지)** 데이터셋은 아래 셀에서 제공됩니다.

In [ ]:
# STEP 3. 전처리 파이프라인 — 선택한 Option에 맞게 직접 작성하세요
# Section04_실습.ipynb의 transform 파이프라인 코드를 참고하세요

# ──────────────────────────────────────────────────────────────────
# 여기에 코드를 작성하세요
# ──────────────────────────────────────────────────────────────────


---
## STEP 4. 모델 구성 ✏️ 직접 작성

### 📖 Option별 참고 노트북

**Option A (분류) → `Section05_실습.ipynb` 실습 2, 3**
- `torchvision.models.resnet50(weights=...)` 로드
- backbone 파라미터 동결 (`requires_grad = False`)
- `model.fc` 를 37 클래스 분류 헤드로 교체
- Forward pass 검증 필수: `assert out.shape == (2, 37)`

**Option B (분할) → `Section08_실습.ipynb` 실습 1, 2**
- `DoubleConv` 블록 정의 (Conv→BN→ReLU × 2)
- `UNet` 클래스 정의 (Encoder + Bottleneck + Decoder + Skip Connection)
- `DiceLoss` 클래스 정의
- Forward pass 검증 필수: `assert out.shape == (2, 3, 128, 128)`

**Option C (탐지) → `Section09_실습.ipynb` 실습 4**
- `fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1)` 로드
- `model.eval().to(device)` 설정
- COCO 클래스: `cat=17, dog=18`

> ⚠️ **Forward pass 검증**: 모델 정의 후 반드시 더미 입력으로 shape를 확인하세요.
> 학습을 시작하기 전에 에러를 잡아야 합니다.

In [ ]:
# STEP 4. 모델 구성 — 선택한 Option에 맞게 직접 작성하세요
# 참고: Section05_실습.ipynb (Option A) / Section08_실습.ipynb (Option B)
#       Section09_실습.ipynb (Option C)

# ──────────────────────────────────────────────────────────────────
# 여기에 코드를 작성하세요
# ──────────────────────────────────────────────────────────────────


---
## STEP 5. 학습 파이프라인 구성 ✏️ 직접 작성

### 📖 Option별 참고 노트북

**Option A (분류) → `Section05_실습.ipynb` + `Section06_실습.ipynb`**

구현해야 할 것:
1. `EarlyStopping` 클래스 (Section06 실습 1)
2. `train_epoch()` 함수 — 학습 루프 4단계 (Section03 실습 3)
3. `eval_epoch()` 함수 — model.eval() + no_grad (Section03 실습 3)
4. **Phase 1**: FC만 학습 (10 에포크, lr=1e-3)
5. **Phase 2**: 전체 레이어 Fine-tuning (Layer-wise LR, 20 에포크)

```python
# Phase 2 Layer-wise LR 구조 (Section05 참고)
optimizer = optim.Adam([
    {'params': model.layer1.parameters(), 'lr': 5e-6},
    ...
    {'params': model.fc.parameters(),     'lr': 1e-4},
])
```

---
**Option B (분할) → `Section08_실습.ipynb` 실습 3**

구현해야 할 것:
1. `EarlyStopping` 클래스
2. `train_epoch_b()` — CE Loss + Dice Loss 합산
3. `eval_epoch_b()` — mIoU 계산 포함
4. 학습 루프 (20 에포크, CosineAnnealingLR)

---
**Option C (탐지) → Section09 실습 4 참고 (추론 전용)**

구현해야 할 것:
1. `compute_det_metrics()` — TP/FP/FN 집계, Precision/Recall/F1 계산
2. Confidence Threshold 실험

> ⚠️ **반드시 포함**: EarlyStopping + LR Scheduler  
> ⚠️ **model.train() / model.eval()** 전환 빠뜨리지 마세요

In [ ]:
# STEP 5. 학습 파이프라인 — 선택한 Option에 맞게 직접 작성하세요
# 참고: Section03_실습.ipynb (학습 루프 구조)
#       Section05_실습.ipynb (전이학습 파이프라인)
#       Section06_실습.ipynb (EarlyStopping, LR Scheduler)
#       Section08_실습.ipynb (분할 학습 루프)

# ──────────────────────────────────────────────────────────────────
# 여기에 코드를 작성하세요
# ──────────────────────────────────────────────────────────────────


---
## STEP 6. 결과 분석

### 📖 템플릿 코드가 아래에 제공됩니다.
학습이 완료된 뒤 실행하세요.  
변수명이 다르다면 본인 코드에 맞게 수정하세요.

> 💡 **분석 체크리스트**
> - Loss·Accuracy 곡선에서 과적합이 발생했는가?
> - 어떤 클래스(품종/영역)에서 오류가 많은가?
> - Grad-CAM(분류)이나 마스크(분할)에서 모델이 올바른 근거로 판단하는가?

In [ ]:
# ─── Option A 결과 분석 템플릿 ───
# 변수명(history_a, model_a 등)을 본인 코드에 맞게 수정하세요

# A-6. 학습 곡선 + 테스트 평가 + Confusion Matrix
# 최적 모델 복원
model_a.load_state_dict(torch.load('best_pet_cls_a.pt', map_location=device))
_, test_acc, all_preds_a, all_labels_a = eval_epoch_a(model_a, test_dl_a, criterion_a, device)
print(f"{'='*40}")
print(f"  최종 테스트 Accuracy: {test_acc:.1%}")
print(f"{'='*40}")

# 학습 곡선
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, len(history_a['train_loss'])+1)
axes[0].plot(ep, history_a['train_loss'], 'b-o', ms=3, lw=2, label='Train')
axes[0].plot(ep, history_a['val_loss'],   'r-s', ms=3, lw=2, label='Val')
axes[0].axvline(x=10, color='gray', lw=1.5, ls='--', alpha=0.6)
axes[0].text(10.3, max(history_a['train_loss'])*0.9, 'Phase2시작', fontsize=8, color='gray')
axes[0].set_title('Loss 곡선', fontweight='bold'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep, [a*100 for a in history_a['train_acc']], 'b-o', ms=3, lw=2, label='Train')
axes[1].plot(ep, [a*100 for a in history_a['val_acc']],   'r-s', ms=3, lw=2, label='Val')
axes[1].axvline(x=10, color='gray', lw=1.5, ls='--', alpha=0.6)
axes[1].set_title('Accuracy 곡선 (%)', fontweight='bold'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Option A — 학습 곡선 (Phase1 + Phase2)', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

# Confusion Matrix (상위 15 클래스)
cm_a = confusion_matrix(all_labels_a, all_preds_a)
top15 = sorted(range(len(CLASS_NAMES)), key=lambda i: cm_a[i,i], reverse=True)[:15]
cm15 = cm_a[np.ix_(top15, top15)]
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(cm15, cmap='Blues')
tnames = [CLASS_NAMES[i].replace('_',' ') for i in top15]
ax.set_xticks(range(15)); ax.set_xticklabels(tnames, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(15)); ax.set_yticklabels(tnames, fontsize=8)
for i in range(15):
    for j in range(15):
        ax.text(j, i, cm15[i,j], ha='center', va='center', fontsize=7,
                color='white' if cm15[i,j]>cm15.max()*0.5 else 'black')
ax.set_title('Confusion Matrix (Accuracy 상위 15 클래스)', fontweight='bold')
plt.colorbar(im); plt.tight_layout(); plt.show()

In [ ]:
# ─── Grad-CAM 시각화 (Option A) ───

# A-7. 오분류 샘플 + Grad-CAM

class GradCAM:
    """CNN이 어디를 보는지 히트맵으로 시각화."""
    def __init__(self, model, target_layer):
        self.model=model; self.gradients=None; self.activations=None
        target_layer.register_forward_hook(lambda m,i,o: setattr(self,'activations',o.detach()))
        target_layer.register_full_backward_hook(lambda m,gi,go: setattr(self,'gradients',go[0].detach()))

    def generate(self, x, class_idx=None):
        self.model.eval()
        out = self.model(x)
        if class_idx is None: class_idx = out.argmax(1).item()
        self.model.zero_grad(); out[0, class_idx].backward()
        w = self.gradients.mean(dim=(2,3), keepdim=True)
        cam = F.relu((w * self.activations).sum(1, keepdim=True))
        cam = F.interpolate(cam, x.shape[2:], mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam-cam.min())/(cam.max()-cam.min()+1e-8)
        return cam, class_idx

gradcam_a = GradCAM(model_a, model_a.layer4[-1].conv3)
MEAN = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
STD  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)

def denorm(t): return (t * STD + MEAN).clamp(0,1).permute(1,2,0).numpy()

# Grad-CAM 시각화 (정분류 / 오분류 4장씩)
model_a.eval()
correct_samples, wrong_samples = [], []
with torch.no_grad():
    for imgs, labels in test_dl_a:
        preds = model_a(imgs.to(device)).argmax(1).cpu()
        for i in range(len(imgs)):
            if preds[i]==labels[i] and len(correct_samples)<4:
                correct_samples.append((imgs[i], labels[i].item(), preds[i].item()))
            if preds[i]!=labels[i] and len(wrong_samples)<4:
                wrong_samples.append((imgs[i], labels[i].item(), preds[i].item()))
        if len(correct_samples)==4 and len(wrong_samples)==4: break

fig, axes = plt.subplots(4, 4, figsize=(15, 14))
titles = ['정분류 — 원본','정분류 — Grad-CAM','오분류 — 원본','오분류 — Grad-CAM']
for col, title in enumerate(titles):
    axes[0][col].set_title(title, fontsize=10, fontweight='bold',
                           color='green' if '정분류' in title else 'red')

for row, (samples, offset) in enumerate([(correct_samples,0),(wrong_samples,2)]):
    for col_pair, (img, true, pred) in enumerate(samples[:2]):  # 각 2장씩 → 실제론 4장
        pass

# 간소화: 정분류 4장 / 오분류 4장
for row, (img, true, pred) in enumerate(correct_samples):
    img_d = denorm(img)
    cam, _ = gradcam_a.generate(img.unsqueeze(0).to(device))
    axes[row][0].imshow(img_d); axes[row][0].set_title(CLASS_NAMES[true].replace('_',' '), fontsize=8, color='green'); axes[row][0].axis('off')
    axes[row][1].imshow(img_d); axes[row][1].imshow(cam, cmap='jet', alpha=0.45); axes[row][1].set_title('Grad-CAM', fontsize=8); axes[row][1].axis('off')

for row, (img, true, pred) in enumerate(wrong_samples):
    img_d = denorm(img)
    cam, _ = gradcam_a.generate(img.unsqueeze(0).to(device))
    axes[row][2].imshow(img_d); axes[row][2].set_title(f"실제:{CLASS_NAMES[true].replace('_',' ')}", fontsize=7, color='red'); axes[row][2].axis('off')
    axes[row][3].imshow(img_d); axes[row][3].imshow(cam, cmap='jet', alpha=0.45)
    axes[row][3].set_title(f"예측:{CLASS_NAMES[pred].replace('_',' ')}", fontsize=7); axes[row][3].axis('off')

plt.suptitle('Option A — Grad-CAM: 정분류 vs 오분류', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ─── Option B 결과 분석 템플릿 ───
# 변수명(model_b, test_ds_b 등)을 본인 코드에 맞게 수정하세요

# B-4. 분할 결과 시각화 (GT vs Prediction)
model_b.load_state_dict(torch.load('best_pet_seg_b.pt', map_location=device))

# 테스트 mIoU
_, test_miou_b = eval_epoch_b(model_b, test_dl_b,
                               criterion_ce_b, criterion_dice_b, device)
print(f"최종 테스트 mIoU: {test_miou_b:.4f}")

# GT vs Pred 시각화
fig, axes = plt.subplots(3, 4, figsize=(14, 11))
CMAP = 'RdYlGn'
model_b.eval()

with torch.no_grad():
    for col in range(4):
        img_t, mask_t = test_ds_b[col*50]
        pred = model_b(img_t.unsqueeze(0).to(device)).argmax(1).squeeze().cpu()
        miou_val, cls_ious = compute_miou(pred, mask_t)

        img_show = (img_t*STD_SEG+MEAN_SEG).clamp(0,1).permute(1,2,0).numpy()

        axes[0][col].imshow(img_show); axes[0][col].set_title(f'이미지 #{col+1}', fontsize=9); axes[0][col].axis('off')
        axes[1][col].imshow(mask_t.numpy(), cmap='tab10', vmin=0, vmax=9)
        axes[1][col].set_title('GT 마스크', fontsize=9); axes[1][col].axis('off')
        axes[2][col].imshow(pred.numpy(), cmap='tab10', vmin=0, vmax=9)
        axes[2][col].set_title(f'예측 (mIoU={miou_val:.3f})', fontsize=9); axes[2][col].axis('off')

plt.suptitle(f'Option B — GT vs Prediction (테스트 mIoU={test_miou_b:.4f})',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

# 학습 곡선
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ep_b = range(1, len(history_b['train_loss'])+1)
axes[0].plot(ep_b, history_b['train_loss'], 'b-o', ms=3, lw=2); axes[0].set_title('Train Loss'); axes[0].grid(alpha=0.3)
axes[1].plot(ep_b, history_b['val_miou'],   'r-o', ms=3, lw=2); axes[1].set_title('Val mIoU');  axes[1].grid(alpha=0.3)
plt.suptitle('Option B — 학습 곡선', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ─── Option C 데이터셋 + 모델 로드 (제공됨) ───

# 1. BBox 파싱용 커스텀 데이터셋
class PetDetectionDataset(Dataset):
    def __init__(self, root, split='trainval', transform=None):
        self.root = root
        self.transform = transform
        self.img_dir = os.path.join(root, 'oxford-iiit-pet', 'images')
        self.anno_dir = os.path.join(root, 'oxford-iiit-pet', 'annotations', 'xmls')

        split_file = os.path.join(root, 'oxford-iiit-pet', 'annotations', f'{split}.txt')
        with open(split_file, 'r') as f:
            self.file_names = [line.strip().split()[0] for line in f.readlines()]

        # XML 파일이 존재하는 이미지만 필터링
        self.file_names = [f for f in self.file_names if os.path.exists(os.path.join(self.anno_dir, f'{f}.xml'))]

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, idx):
        file_name = self.file_names[idx]
        img_path = os.path.join(self.img_dir, f'{file_name}.jpg')
        img = Image.open(img_path).convert("RGB")

        xml_path = os.path.join(self.anno_dir, f'{file_name}.xml')
        tree = ET.parse(xml_path)
        root_xml = tree.getroot()

        bndbox = root_xml.find('object').find('bndbox')
        xmin = int(bndbox.find('xmin').text)
        ymin = int(bndbox.find('ymin').text)
        xmax = int(bndbox.find('xmax').text)
        ymax = int(bndbox.find('ymax').text)

        boxes = torch.tensor([[xmin, ymin, xmax, ymax]], dtype=torch.float32)
        target = {
            "boxes": boxes,
            "labels": torch.ones((1,), dtype=torch.int64), # 단일 클래스(머리=1)
            "image_id": torch.tensor([idx])
        }

        if self.transform is not None:
            img = self.transform(img)

        return img, target

# =====================================================================
# C-1. 탐지용 데이터셋 + Faster R-CNN 로드
# =====================================================================
DATA_ROOT = './data'

# ⚠️ 'test' 스플릿은 XML이 없으므로 'trainval'을 사용합니다.
base_det_ds = PetDetectionDataset(root=DATA_ROOT, split='trainval', transform=transforms.ToTensor())

# 전체 데이터 중 마지막 200장을 테스트 세트로 분리하여 활용
total_len = len(base_det_ds)
test_ds_c = Subset(base_det_ds, range(total_len - 200, total_len))

# Faster R-CNN (COCO 사전학습)
weights_c = FasterRCNN_ResNet50_FPN_Weights.COCO_V1
detector_c = fasterrcnn_resnet50_fpn(weights=weights_c)

# device 변수가 할당되어 있어야 합니다 (cpu 또는 cuda)
if 'device' not in globals():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
detector_c.eval().to(device)

COCO_LABELS = weights_c.meta['categories']
CAT_IDX, DOG_IDX = 17, 18
PET_INDICES = {CAT_IDX, DOG_IDX}

print(f"✅ 탐지 모델 로드 완료")
print(f"✅ 테스트 세트: {len(test_ds_c)}장")
print(f"✅ 관심 클래스: cat({CAT_IDX}), dog({DOG_IDX})")

In [ ]:
# ─── Option C BBox 시각화 템플릿 ───

# C-2. BBox 시각화 + 추론 결과 확인

def visualize_detection(img_t, gt_bbox, pred_boxes, pred_scores, pred_labels,
                         threshold=0.5, ax=None, title=''):
    """이미지에 GT BBox(파란색)와 예측 BBox(빨간색)를 오버레이."""
    if ax is None: fig, ax = plt.subplots()
    img_np = img_t.cpu().permute(1,2,0).numpy() # gpu 텐서일 경우 대비 cpu() 추가
    ax.imshow(img_np)
    h, w = img_np.shape[:2]

    # GT BBox (파란색)
    if isinstance(gt_bbox, (list, tuple)) and len(gt_bbox)==4:
        xmin,ymin,xmax,ymax = gt_bbox
        ax.add_patch(patches.Rectangle((xmin,ymin),(xmax-xmin),(ymax-ymin),
            lw=2.5, edgecolor='blue', facecolor='none'))
        ax.text(xmin, ymin-5, 'GT', color='blue', fontsize=8, fontweight='bold')

    # 예측 BBox (빨간색, threshold 이상만)
    for box, score, label in zip(pred_boxes, pred_scores, pred_labels):
        if score < threshold: continue
        if label.item() not in PET_INDICES: continue
        x1,y1,x2,y2 = box.tolist()
        ax.add_patch(patches.Rectangle((x1,y1),(x2-x1),(y2-y1),
            lw=2, edgecolor='red', facecolor='none'))
        ax.text(x1, y1-5, f"{COCO_LABELS[label.item()]}:{score:.2f}",
                color='red', fontsize=7, fontweight='bold')

    ax.set_title(title, fontsize=8); ax.axis('off')

# 추론 + 시각화 (4장)
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle('Option C — Faster R-CNN 탐지 결과(파란=GT, 빨간=예측)', fontsize=11, fontweight='bold')

with torch.no_grad():
    for col in range(4):
        # 1. C-1 데이터셋은 bbox를 딕셔너리 형태로 반환합니다.
        img_t, target = test_ds_c[col*45]

        # 2. 딕셔너리에서 좌표만 리스트로 추출! (이 부분이 핵심 해결책)
        gt_bbox_list = target['boxes'][0].tolist()

        pred = detector_c([img_t.to(device)])[0]

        visualize_detection(
            img_t,
            gt_bbox_list, # <--- 추출한 리스트를 전달
            pred['boxes'].cpu(), pred['scores'].cpu(), pred['labels'].cpu(),
            threshold=0.5, ax=axes[col], title=f'이미지 #{col+1}'
        )

plt.tight_layout(); plt.show()

In [ ]:
# ─── Option C Threshold 분석 템플릿 ───
# compute_det_metrics() 함수를 직접 작성한 뒤 실행하세요

# =====================================================================
# C-4. Confidence Threshold 변화에 따른 Precision-Recall 트레이드오프
# =====================================================================

thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
precs, recs = [], []

# ⚠️ 여기서 호출되는 compute_det_metrics는 반드시 '수정된 C-3' 셀이 실행된 상태여야 합니다.
for th in tqdm(thresholds, desc="Threshold", ncols=80):
    m = compute_det_metrics(test_ds_c, detector_c, device,
                             iou_thresh=0.5, conf_thresh=th)
    precs.append(m['precision'])
    recs.append(m['recall'])
    print(f"  conf_thresh={th:.1f} → Precision={m['precision']:.3f}, Recall={m['recall']:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Threshold별 Precision / Recall 변화
axes[0].plot(thresholds, precs, 'b-o', ms=6, lw=2, label='Precision')
axes[0].plot(thresholds, recs,  'r-s', ms=6, lw=2, label='Recall')
axes[0].set_xlabel('Confidence Threshold')
axes[0].set_title('Threshold별 Precision·Recall 변화', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_ylim(0, 1.05)

# PR 곡선
axes[1].plot(recs, precs, 'g-o', ms=6, lw=2)
for i, th in enumerate(thresholds):
    axes[1].annotate(f'th={th}', (recs[i], precs[i]),
                     textcoords='offset points', xytext=(5,3), fontsize=7)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall 곡선', fontweight='bold')
axes[1].set_xlim(0, 1.05)
axes[1].set_ylim(0, 1.05)
axes[1].grid(alpha=0.3)

plt.suptitle('Option C — Confidence Threshold 분석', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 threshold를 높이면 Precision↑ Recall↓ — 상충 관계(Trade-off)가 뚜렷하게 관찰됩니다.")

In [ ]:
# ─── 종합 결과 비교 (제공됨) ───
# 변수명을 본인 코드에 맞게 수정하세요

# =====================================================================
# 세 가지 Option 성능 종합 정리
# =====================================================================

print("\n" + "="*55)
print("  MiniProject — Option 별 최종 성능 요약")
print("="*55)
# (주의) test_acc와 test_miou_b는 Option A와 B 셀을 실행하여 메모리에 저장되어 있어야 합니다.
print(f"  Option A (분류)  | Test Accuracy : {test_acc:.1%}")
print(f"  Option B (분할)  | Test mIoU     : {test_miou_b:.4f}")
print(f"  Option C (탐지)  | Precision     : {metrics_c['precision']:.4f}")
print(f"                   | Recall        : {metrics_c['recall']:.4f}")
print(f"                   | F1-Score      : {metrics_c['f1']:.4f}")
print("="*55)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 그래프 그릴 데이터 리스트: (제목, 값, 색상, y축 최대치)
plot_data = [
    ("Option A\n품종 분류\nAccuracy", test_acc * 100, '#81c784', 100),
    ("Option B\n전경/배경 분할\nmIoU", test_miou_b, '#4fc3f7', 1),
    ("Option C\n머리 탐지\nF1-Score", metrics_c['f1'], '#ffb74d', 1)
]

for ax, (title, value, color, ymax) in zip(axes, plot_data):
    ax.bar(['결과'], [value], color=color, alpha=0.85, width=0.4, edgecolor='white', lw=1.5)
    ax.set_ylim(0, ymax)
    ax.set_title(title, fontsize=11, fontweight='bold')

    # 막대 위에 수치 텍스트 표시
    text_val = f'{value:.3f}' if ymax == 1 else f'{value:.1f}%'
    ax.text(0, value + ymax * 0.02, text_val,
            ha='center', fontsize=14, fontweight='bold', color=color)

    ax.set_xticks([])
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('MiniProject — 세 가지 Option 최종 성능', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🔍 학습 완료 모델로 커스텀 이미지 추론

아래 코드는 제공됩니다. 원하는 이미지로 테스트해보세요.

| Option | 설정 변수 | 기본값 |
|--------|----------|--------|
| A (분류) | `USE_URL_A`, `IMAGE_PATH_A`, `TOP_K_A` | URL 이미지 사용 |
| B (분할) | `USE_URL_B`, `IMAGE_PATH_B` | URL 이미지 사용 |
| C (탐지) | `USE_URL_C`, `CONF_THRESH_C` | URL 이미지 사용 |

In [ ]:
# [Option A] 커스텀 이미지 품종 분류
# best_pet_cls_a.pt 가중치 필요
import urllib.request
from PIL import Image

USE_URL_A    = True
IMAGE_PATH_A = "./my_pet.jpg"
IMAGE_URL_A  = "https://cdn.kormedi.com/wp-content/uploads/2022/03/eab095ec9584eca780-580x405.jpg.webp"
TOP_K_A      = 5

if USE_URL_A:
    urllib.request.urlretrieve(IMAGE_URL_A, "/tmp/infer_a.jpg")
    img_pil_a = Image.open("/tmp/infer_a.jpg").convert("RGB")
else:
    img_pil_a = Image.open(IMAGE_PATH_A).convert("RGB")
    print(f"파일 로드: {IMAGE_PATH_A}  크기: {img_pil_a.size}")

infer_tf_a = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])

model_a.load_state_dict(torch.load('best_pet_cls_a.pt', map_location=device))
model_a.eval()
img_t_a = infer_tf_a(img_pil_a).unsqueeze(0).to(device)

with torch.no_grad():
    probs_a = torch.softmax(model_a(img_t_a), dim=1)
    top_probs_a, top_idx_a = probs_a.topk(TOP_K_A, dim=1)

pred_name_a = CLASS_NAMES[top_idx_a[0][0].item()].replace('_', ' ')
pred_prob_a = top_probs_a[0][0].item()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(img_pil_a.resize((224,224)))
axes[0].set_title(f"예측: {pred_name_a}\n신뢰도: {pred_prob_a:.1%}",
                  fontsize=12, fontweight='bold',
                  color='#2ecc71' if pred_prob_a > 0.5 else '#e74c3c')
axes[0].axis('off')

top_names_a  = [CLASS_NAMES[i.item()].replace('_',' ') for i in top_idx_a[0]]
top_values_a = [p.item()*100 for p in top_probs_a[0]]
colors_a     = ['#2ecc71'] + ['#3498db']*(TOP_K_A-1)
axes[1].barh(range(TOP_K_A)[::-1], top_values_a, color=colors_a, alpha=0.85, edgecolor='white')
axes[1].set_yticks(range(TOP_K_A)[::-1])
axes[1].set_yticklabels(top_names_a, fontsize=10)
axes[1].set_xlabel('확률 (%)')
axes[1].set_title(f'Top-{TOP_K_A} 후보', fontweight='bold')
axes[1].set_xlim(0, max(top_values_a)*1.3)
axes[1].grid(axis='x', alpha=0.3)
for i, val in enumerate(top_values_a[::-1]):
    axes[1].text(val+0.5, i, f'{val:.1f}%', va='center', fontsize=9, fontweight='bold')

gradcam_a_infer = GradCAM(model_a, model_a.layer4[-1].conv3)
cam_a, _ = gradcam_a_infer.generate(img_t_a)
img_224_a = np.array(img_pil_a.resize((224,224))) / 255.0
axes[2].imshow(img_224_a)
axes[2].imshow(cam_a, cmap='jet', alpha=0.45)
axes[2].set_title('Grad-CAM — 모델 집중 영역', fontweight='bold')
axes[2].axis('off')

plt.suptitle('[Option A] 품종 분류 추론 결과', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

print(f"\n{'='*45}")
print(f"  예측 품종: {pred_name_a}")
print(f"  신뢰도:   {pred_prob_a:.1%}")
print('─'*45)
for rank, (name, prob) in enumerate(zip(top_names_a, top_values_a), 1):
    bar = '█'*int(prob/4) + '░'*(25-int(prob/4))
    print(f"  {rank}위 {name:<25} {bar} {prob:.1f}%")
print(f"{'='*45}")

In [ ]:
# [Option B] 커스텀 이미지 전경/배경 분할
# best_pet_seg_b.pt 가중치 필요
import urllib.request
from PIL import Image

USE_URL_B    = True
IMAGE_PATH_B = "./my_pet.jpg"
IMAGE_URL_B  = "https://cdn.kormedi.com/wp-content/uploads/2022/03/eab095ec9584eca780-580x405.jpg.webp"

if USE_URL_B:
    urllib.request.urlretrieve(IMAGE_URL_B, "/tmp/infer_b.jpg")
    img_pil_b = Image.open("/tmp/infer_b.jpg").convert("RGB")
else:
    img_pil_b = Image.open(IMAGE_PATH_B).convert("RGB")
    print(f"파일 로드: {IMAGE_PATH_B}  크기: {img_pil_b.size}")

MEAN_B = [0.485,0.456,0.406]; STD_B = [0.229,0.224,0.225]
infer_tf_b = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN_B, STD_B),
])

model_b.load_state_dict(torch.load('best_pet_seg_b.pt', map_location=device))
model_b.eval()
img_t_b = infer_tf_b(img_pil_b).unsqueeze(0).to(device)

with torch.no_grad():
    pred_mask_b = model_b(img_t_b).argmax(dim=1).squeeze().cpu()

total_px = pred_mask_b.numel()
fg_ratio = (pred_mask_b == 0).sum().item() / total_px * 100
bg_ratio = (pred_mask_b == 1).sum().item() / total_px * 100
bd_ratio = (pred_mask_b == 2).sum().item() / total_px * 100

color_mask_b = np.zeros((*pred_mask_b.shape, 3), dtype=np.uint8)
for cls, color in [(0,[231,76,60]), (1,[52,152,219]), (2,[243,156,18])]:
    color_mask_b[pred_mask_b.numpy()==cls] = color

img_vis_b = (img_t_b.squeeze().cpu() *
             torch.tensor(STD_B).view(3,1,1) +
             torch.tensor(MEAN_B).view(3,1,1)).clamp(0,1).permute(1,2,0).numpy()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(img_vis_b)
axes[0].set_title('원본 이미지', fontsize=12, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(color_mask_b)
from matplotlib.patches import Patch
lgd = [Patch(color='#e74c3c', label=f'전경(동물) {fg_ratio:.1f}%'),
       Patch(color='#3498db', label=f'배경 {bg_ratio:.1f}%'),
       Patch(color='#f39c12', label=f'경계 {bd_ratio:.1f}%')]
axes[1].legend(handles=lgd, loc='lower right', fontsize=8)
axes[1].set_title('예측 마스크', fontsize=12, fontweight='bold')
axes[1].axis('off')

overlay_b = img_vis_b.copy()
fg_mask = (pred_mask_b.numpy() == 0)
overlay_b[fg_mask] = overlay_b[fg_mask]*0.5 + np.array([0.9,0.2,0.2])*0.5
axes[2].imshow(overlay_b)
axes[2].set_title('전경 오버레이\n(빨간색=동물 영역)', fontsize=11, fontweight='bold')
axes[2].axis('off')

plt.suptitle('[Option B] 전경/배경 분할 추론 결과', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

print(f"\n{'='*40}")
print(f"  전경(동물): {fg_ratio:.1f}%")
print(f"  배경:       {bg_ratio:.1f}%")
print(f"  경계:       {bd_ratio:.1f}%")
print(f"{'='*40}")

In [ ]:
# [Option C] 커스텀 이미지 객체 탐지
# COCO 사전학습 모델 사용 (별도 저장 파일 불필요)
import urllib.request
from PIL import Image

USE_URL_C     = True
IMAGE_PATH_C  = "./my_pet.jpg"
IMAGE_URL_C   = "https://cdn.kormedi.com/wp-content/uploads/2022/03/eab095ec9584eca780-580x405.jpg.webp"
CONF_THRESH_C = 0.4
NMS_THRESH_C  = 0.5

if USE_URL_C:
    urllib.request.urlretrieve(IMAGE_URL_C, "/tmp/infer_c.jpg")
    img_pil_c = Image.open("/tmp/infer_c.jpg").convert("RGB")
else:
    img_pil_c = Image.open(IMAGE_PATH_C).convert("RGB")
    print(f"파일 로드: {IMAGE_PATH_C}  크기: {img_pil_c.size}")

img_t_c = transforms.ToTensor()(img_pil_c).to(device)
detector_c.eval()
with torch.no_grad():
    pred_c = detector_c([img_t_c])[0]

mask_c   = pred_c['scores'] > CONF_THRESH_C
boxes_c  = pred_c['boxes'][mask_c].cpu()
scores_c = pred_c['scores'][mask_c].cpu()
labels_c = pred_c['labels'][mask_c].cpu()

if len(boxes_c) > 0:
    keep     = nms(boxes_c, scores_c, iou_threshold=NMS_THRESH_C)
    boxes_c  = boxes_c[keep]
    scores_c = scores_c[keep]
    labels_c = labels_c[keep]

print(f"탐지된 객체 수: {len(boxes_c)}개")

img_np_c   = np.array(img_pil_c) / 255.0
PALETTE_C  = plt.cm.Set1(np.linspace(0, 1, 10))

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].imshow(img_np_c)
for box, score, label in zip(boxes_c, scores_c, labels_c):
    x1,y1,x2,y2 = box.tolist()
    color    = PALETTE_C[label.item() % 10]
    cls_name = COCO_LABELS[label.item()] if label.item() < len(COCO_LABELS) else str(label.item())
    rect = patches.Rectangle((x1,y1),(x2-x1),(y2-y1), lw=2.5, edgecolor=color, facecolor='none')
    axes[0].add_patch(rect)
    axes[0].text(x1, max(y1-6,0), f"{cls_name}: {score:.2f}", color='white',
                  fontsize=8.5, fontweight='bold',
                  bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.85))
axes[0].set_title(f'탐지 결과 — {len(boxes_c)}개 객체', fontsize=12, fontweight='bold')
axes[0].axis('off')

if len(boxes_c) > 0:
    det_names  = [COCO_LABELS[l.item()] if l.item()<len(COCO_LABELS) else str(l.item()) for l in labels_c]
    det_scores = [s.item()*100 for s in scores_c]
    det_colors = [PALETTE_C[l.item()%10] for l in labels_c]
    y_pos = range(len(det_names))
    axes[1].barh(list(y_pos)[::-1], det_scores, color=det_colors, alpha=0.85, edgecolor='white')
    axes[1].set_yticks(list(y_pos)[::-1])
    axes[1].set_yticklabels([f"#{i+1} {n}" for i,n in enumerate(det_names)], fontsize=10)
    axes[1].set_xlabel('신뢰도 (%)')
    axes[1].set_title('탐지 객체별 신뢰도', fontsize=12, fontweight='bold')
    axes[1].set_xlim(0, 115); axes[1].grid(axis='x', alpha=0.3)
    for i, val in enumerate(det_scores[::-1]):
        axes[1].text(val+0.5, i, f'{val:.1f}%', va='center', fontsize=9, fontweight='bold')
else:
    axes[1].text(0.5, 0.5, '탐지된 객체 없음\n\nconf_thresh를 낮춰보세요',
                  ha='center', va='center', transform=axes[1].transAxes, fontsize=13, color='gray')
    axes[1].axis('off')

plt.suptitle('[Option C] 객체 탐지 추론 결과', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

print(f"\n{'='*50}")
for i,(box,score,label) in enumerate(zip(boxes_c,scores_c,labels_c)):
    x1,y1,x2,y2 = [int(v) for v in box.tolist()]
    cls = COCO_LABELS[label.item()] if label.item()<len(COCO_LABELS) else str(label.item())
    print(f"  #{i+1} {cls:<15} 신뢰도:{score:.1%}  BBox:[{x1},{y1},{x2},{y2}]")
print(f"{'='*50}")

---
## STEP 7. 회고 ✏️ 직접 작성

### 📊 나의 실험 결과 요약

| 항목 | 내용 |
|------|------|
| 선택한 태스크 | |
| 사용한 모델 | |
| 최종 Train 성능 | |
| 최종 Val 성능 | |
| 최종 Test 성능 | |
| 총 학습 에포크 수 | |
| EarlyStopping 발동 여부 | |

---

### ✅ 잘 된 점
1.
2.
3.

---

### 🔧 아쉬운 점 / 개선하고 싶은 것
1.
2.
3.

---

### 🚀 다음에 시도해볼 것
-
-
-